# NBA DFS Backtest Evaluation & Visualization

This notebook loads saved backtest results and generates interactive visualizations.

## Prerequisites
Run `run_backtest.ipynb` first to generate prediction outputs.

## Workflow
1. Select which backtest run to evaluate (by timestamp)
2. Load predictions and actuals from parquet files
3. Reconstruct performance metrics
4. Generate interactive Altair visualizations

## Visualizations Included
- Performance over time (MAPE, RMSE, Correlation)
- Model vs Benchmark comparison
- Salary tier analysis
- Error distributions
- Comprehensive dashboards

## Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluation.altair_visualizations import AltairVisualizer
from src.evaluation.metrics.accuracy import MAPEMetric, RMSEMetric, MAEMetric, CorrelationMetric

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Setup complete')

Setup complete


## Configuration - Select Run to Evaluate

In [2]:
# Output directory where backtest results are saved
from src.config.paths import OUTPUTS_DIR, PROJECT_ROOT
OUTPUT_DIR = str(OUTPUTS_DIR)


# Select which run to evaluate
# Set to None to use most recent run, or specify timestamp like '20251019_050018'
RUN_TIMESTAMP = '20251022_054645'

# Find available runs
output_path = Path(OUTPUT_DIR)
run_dirs = sorted([d for d in output_path.iterdir() if d.is_dir()], reverse=True)

if not run_dirs:
    print(f"ERROR: No backtest runs found in {OUTPUT_DIR}")
    print(f"Run 'run_backtest.ipynb' first to generate results")
else:
    print(f"Found {len(run_dirs)} backtest run(s):\n")
    for i, run_dir in enumerate(run_dirs[:100]):
        # Get run timestamp
        timestamp = run_dir.name
        
        # Count prediction files
        pred_dir = run_dir / 'predictions'
        if pred_dir.exists():
            num_files = len(list(pred_dir.glob('*_with_actuals.parquet')))
            print(f"  {i+1}. {timestamp} ({num_files} slates)")
        else:
            print(f"  {i+1}. {timestamp} (no predictions)")
    
    if len(run_dirs) > 10:
        print(f"  ... and {len(run_dirs) - 10} more")
    
    print(f"\nSet RUN_TIMESTAMP to select a specific run, or leave None for most recent")

Found 26 backtest run(s):

  1. altair_charts (no predictions)
  2. 20251022_071949 (0 slates)
  3. 20251022_071340 (0 slates)
  4. 20251022_071003 (0 slates)
  5. 20251022_063618 (0 slates)
  6. 20251022_063327 (0 slates)
  7. 20251022_054645 (0 slates)
  8. 20251022_054428 (0 slates)
  9. 20251022_053919 (0 slates)
  10. 20251022_041319 (0 slates)
  11. 20251022_032232 (0 slates)
  12. 20251022_022639 (0 slates)
  13. 20251022_021258 (0 slates)
  14. 20251022_015951 (0 slates)
  15. 20251022_015913 (0 slates)
  16. 20251022_014535 (0 slates)
  17. 20251022_014527 (0 slates)
  18. 20251022_001025 (0 slates)
  19. 20251021_214641 (0 slates)
  20. 20251021_214511 (0 slates)
  21. 20251020_225001 (5 slates)
  22. 20251020_215437 (19 slates)
  23. 20251020_212758 (9 slates)
  24. 20251020_205622 (5 slates)
  25. 20251020_205057 (5 slates)
  26. 20251020_052352 (75 slates)
  ... and 16 more

Set RUN_TIMESTAMP to select a specific run, or leave None for most recent


## Load Backtest Results

In [3]:
# Select run to load
if RUN_TIMESTAMP:
    selected_run = output_path / RUN_TIMESTAMP
    if not selected_run.exists():
        print(f"ERROR: Run {RUN_TIMESTAMP} not found")
        selected_run = None
else:
    selected_run = run_dirs[0] if run_dirs else None

if selected_run is None:
    print("No valid run selected. Check configuration above.")
else:
    predictions_dir = selected_run / 'predictions'
    
    if not predictions_dir.exists():
        print(f"ERROR: No predictions directory in {selected_run.name}")
    else:
        print(f"Loading results from: {selected_run.name}")
        print(f"Location: {selected_run}\n")
        
        # Load all slate results
        actuals_files = sorted(predictions_dir.glob('*_with_actuals.parquet'))
        
        if not actuals_files:
            print("ERROR: No result files found in predictions directory")
        else:
            print(f"Loading {len(actuals_files)} slate results...")
            
            # Load all predictions
            all_predictions_list = []
            for file in actuals_files:
                df = pd.read_parquet(file)
                # Add date column if missing (for backward compatibility with older backtest runs)
                if 'date' not in df.columns:
                    # Extract date from filename: {date}_with_actuals.parquet
                    date_str = file.stem.replace('_with_actuals', '')
                    df['date'] = date_str
                all_predictions_list.append(df)
            
            all_predictions_df = pd.concat(all_predictions_list, ignore_index=True)
            print(f"Loaded {len(all_predictions_df)} total predictions\n")
            
            # Initialize metrics
            mape_metric = MAPEMetric()
            rmse_metric = RMSEMetric()
            mae_metric = MAEMetric()
            corr_metric = CorrelationMetric()
            
            # Reconstruct daily results
            print("Calculating daily metrics...")
            daily_results = []
            
            for date in sorted(all_predictions_df['date'].unique()):
                date_df = all_predictions_df[all_predictions_df['date'] == date]
                
                # Filter out near-zero actuals to prevent MAPE explosion (division by zero)
                # Players with <5 fpts are typically DNP/injury/ejection cases
                mape_filter = date_df['actual_fpts'] >= 5.0
                
                # Calculate model MAPE with filtering
                if mape_filter.any():
                    model_mape = mape_metric.calculate(
                        date_df.loc[mape_filter, 'actual_fpts'],
                        date_df.loc[mape_filter, 'projected_fpts']
                    )
                else:
                    model_mape = np.nan
                
                # Calculate benchmark MAPE with same filters
                has_benchmark = (date_df['benchmark_pred'] > 0) & (date_df['actual_fpts'] >= 5.0)
                if has_benchmark.any():
                    benchmark_mape = mape_metric.calculate(
                        date_df[has_benchmark]['actual_fpts'], 
                        date_df[has_benchmark]['benchmark_pred']
                    )
                    benchmark_rmse = rmse_metric.calculate(
                        date_df[has_benchmark]['actual_fpts'],
                        date_df[has_benchmark]['benchmark_pred']
                    )
                else:
                    benchmark_mape = np.nan
                    benchmark_rmse = np.nan
                
                daily_results.append({
                    'date': date,
                    'num_players': len(date_df),
                    'model_mape': model_mape,
                    'model_rmse': rmse_metric.calculate(date_df['actual_fpts'], date_df['projected_fpts']),
                    'model_mae': mae_metric.calculate(date_df['actual_fpts'], date_df['projected_fpts']),
                    'model_corr': corr_metric.calculate(date_df['actual_fpts'], date_df['projected_fpts']),
                    'benchmark_mape': benchmark_mape,
                    'benchmark_rmse': benchmark_rmse,
                    'mean_actual': date_df['actual_fpts'].mean(),
                    'mean_projected': date_df['projected_fpts'].mean(),
                    'mean_benchmark': date_df['benchmark_pred'].mean()
                })
            
            results_df = pd.DataFrame(daily_results)
            
            # Create results summary dict
            results = {
                'num_slates': len(results_df),
                'date_range': f"{results_df['date'].min()} to {results_df['date'].max()}",
                'total_players_evaluated': results_df['num_players'].sum(),
                'avg_players_per_slate': results_df['num_players'].mean(),
                'model_mean_mape': results_df['model_mape'].mean(),
                'model_median_mape': results_df['model_mape'].median(),
                'model_std_mape': results_df['model_mape'].std(),
                'model_mean_rmse': results_df['model_rmse'].mean(),
                'model_mean_mae': results_df['model_mae'].mean(),
                'model_mean_correlation': results_df['model_corr'].mean(),
                'benchmark_mean_mape': results_df['benchmark_mape'].mean(),
                'benchmark_median_mape': results_df['benchmark_mape'].median(),
                'mape_improvement': results_df['benchmark_mape'].mean() - results_df['model_mape'].mean(),
                'daily_results': results_df,
                'all_predictions': all_predictions_df
            }
            
            print(f"\n{'='*80}")
            print('RESULTS LOADED')
            print('='*80)
            print(f"Slates: {results['num_slates']}")
            print(f"Date range: {results['date_range']}")
            print(f"Total players: {results['total_players_evaluated']:.0f}")
            print(f"Model MAPE: {results['model_mean_mape']:.2f}%")
            print(f"Benchmark MAPE: {results['benchmark_mean_mape']:.2f}%")
            print(f"Improvement: {results['mape_improvement']:+.2f}%")
            print(f"\nNote: MAPE calculated only for players with actual_fpts >= 5.0")
            print(f"Data loaded successfully. Proceed to visualizations below.")

Loading results from: 20251022_054645
Location: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs\20251022_054645

ERROR: No result files found in predictions directory


## Summary Statistics

In [4]:
print('='*80)
print('BACKTEST RESULTS SUMMARY')
print('='*80)
print(f'\nNumber of Slates: {results["num_slates"]}')
print(f'Date Range: {results["date_range"]}')
print(f'\nTotal Players Evaluated: {results["total_players_evaluated"]:.0f}')
print(f'Average Players per Slate: {results["avg_players_per_slate"]:.1f}')
print(f'\nModel Performance:')
print(f'  Mean MAPE: {results["model_mean_mape"]:.2f}%')
print(f'  Median MAPE: {results["model_median_mape"]:.2f}%')
print(f'  Std MAPE: {results["model_std_mape"]:.2f}%')
print(f'  Mean RMSE: {results["model_mean_rmse"]:.2f}')
print(f'  Mean MAE: {results["model_mean_mae"]:.2f}')
print(f'  Mean Correlation: {results["model_mean_correlation"]:.3f}')
print(f'\nBenchmark Performance:')
print(f'  Mean MAPE: {results["benchmark_mean_mape"]:.2f}%')
print(f'  Median MAPE: {results["benchmark_median_mape"]:.2f}%')
print(f'\nImprovement (Model vs Benchmark):')
print(f'  MAPE Improvement: {results["mape_improvement"]:+.2f}%')

BACKTEST RESULTS SUMMARY


NameError: name 'results' is not defined

## Performance Metrics Dashboard

Interactive visualizations of MAPE, RMSE, correlation, and player counts over time.

In [ ]:
# Initialize visualizer with dark theme
viz = AltairVisualizer(output_dir=OUTPUTS_DIR, theme='dark')

# Prepare data
results_df_viz = results_df.copy()
results_df_viz['slate_number'] = range(len(results_df_viz))

# MAPE over time (model vs benchmark)
mape_data = results_df_viz[['slate_number', 'date', 'model_mape', 'benchmark_mape']]
mape_chart = viz.create_multi_line_chart(
    data=mape_data,
    x='date',
    y_columns=['model_mape', 'benchmark_mape'],
    title='MAPE Over Time - Model vs Benchmark',
    width=450,
    height=250
)

# RMSE over time
rmse_chart = viz.create_line_chart(
    data=results_df_viz,
    x='date',
    y='model_rmse',
    title='RMSE Over Time',
    width=450,
    height=250
)

# Correlation over time
corr_chart = viz.create_line_chart(
    data=results_df_viz,
    x='date',
    y='model_corr',
    title='Correlation Over Time',
    width=450,
    height=250
)

# Players evaluated per slate
players_chart = viz.create_bar_chart(
    data=results_df_viz,
    x='date',
    y='num_players',
    title='Players Evaluated Per Slate',
    width=450,
    height=250
)

# Combine into 2x2 dashboard
dashboard = viz.create_dashboard([mape_chart, rmse_chart, corr_chart, players_chart], columns=2)
dashboard

alt.VConcatChart(...)

## Salary Tier Analysis

Performance breakdown by salary tiers (if available in results).

In [ ]:
if 'tier_comparison' in results:
    tier_comparison = results['tier_comparison']
    
    # MAPE by salary tier (grouped bar)
    tier_data = tier_comparison[['salary_tier', 'model_mape', 'benchmark_mape']].melt(
        id_vars='salary_tier',
        value_vars=['model_mape', 'benchmark_mape'],
        var_name='metric',
        value_name='mape'
    )
    
    tier_mape_chart = viz.create_bar_chart(
        data=tier_data,
        x='salary_tier',
        y='mape',
        color='metric',
        title='MAPE by Salary Tier',
        width=450,
        height=350
    )
    
    # Improvement chart
    improvement_chart = viz.create_bar_chart(
        data=tier_comparison,
        x='salary_tier',
        y='mape_improvement',
        color='mape_improvement',
        title='Model Improvement Over Benchmark',
        width=450,
        height=350
    )
    
    # Combine
    tier_dashboard = viz.create_dashboard([tier_mape_chart, improvement_chart], columns=2)
    tier_dashboard
else:
    print('Tier comparison not available in this backtest run.')
    print('Salary tier analysis requires SALARY_TIERS configuration in run_backtest.ipynb')

Tier comparison not available in this backtest run.
Salary tier analysis requires SALARY_TIERS configuration in run_backtest.ipynb


## Comprehensive Backtest Dashboard

Uses built-in dashboard method to create overview of all key metrics.

In [ ]:
# Use built-in backtest dashboard method
backtest_dashboard = viz.create_backtest_dashboard(results)
backtest_dashboard

alt.VConcatChart(...)

## Error Analysis

Model vs benchmark error comparison with scatter plot and distribution histogram.

In [ ]:
# Filter for valid comparisons
try:
    comparison_df = all_predictions_df[
        (all_predictions_df['projected_fpts'] > 0) & 
        (all_predictions_df['benchmark_pred'] > 0)
    ].copy()

    # Calculate errors
    comparison_df['model_error'] = np.abs(comparison_df['projected_fpts'] - comparison_df['actual_fpts'])
    comparison_df['benchmark_error'] = np.abs(comparison_df['benchmark_pred'] - comparison_df['actual_fpts'])
    comparison_df['error_diff'] = comparison_df['benchmark_error'] - comparison_df['model_error']

    print(f"Comparing {len(comparison_df)} predictions")
    print(f"Mean error difference: {comparison_df['error_diff'].mean():.2f}")
    print(f"Positive values (model better): {(comparison_df['error_diff'] > 0).sum()} ({(comparison_df['error_diff'] > 0).sum() / len(comparison_df) * 100:.1f}%)")

    # Model vs Benchmark error scatter
    error_scatter = viz.create_scatter_plot(
        data=comparison_df,
        x='benchmark_error',
        y='model_error',
        title='Model vs Benchmark Error Comparison',
        width=500,
        height=400
    )

    # Error difference histogram
    error_hist = viz.create_histogram(
        data=comparison_df,
        column='error_diff',
        bins=30,
        title='Error Difference Distribution (Positive = Model Better)',
        width=500,
        height=400,
        color='#18FF6D'
    )

    # Combine
    error_dashboard = viz.create_dashboard([error_scatter, error_hist], columns=2)
    error_dashboard
    
except Exception as e:
    print(f"Error creating error analysis dashboard: {e}")
    print("Continuing with other visualizations...")

## Actual vs Predicted Scatter

Direct comparison of actual vs predicted fantasy points with trend line.

In [ ]:
# Actual vs Predicted scatter plot
actual_vs_pred = viz.create_scatter_plot(
    data=all_predictions_df,
    x='actual_fpts',
    y='projected_fpts',
    title='Actual vs Predicted Fantasy Points',
    width=600,
    height=500
)

actual_vs_pred

alt.LayerChart(...)

## Save Visualizations

Optionally save charts to HTML files for sharing or reporting.

In [ ]:
# Save visualizations (safe version)
print("Saving visualizations...")

try:
    if 'dashboard' in locals():
        viz.save_chart(dashboard, f'performance_dashboard_{selected_run.name}')
        print(f"✓ Saved performance_dashboard_{selected_run.name}")
except Exception as e:
    print(f"✗ Could not save performance_dashboard: {e}")

try:
    if 'error_dashboard' in locals():
        viz.save_chart(error_dashboard, f'error_analysis_{selected_run.name}')
        print(f"✓ Saved error_analysis_{selected_run.name}")
    else:
        print("Note: error_dashboard not available (error analysis skipped)")
except Exception as e:
    print(f"✗ Could not save error_dashboard: {e}")

try:
    if 'backtest_dashboard' in locals():
        viz.save_chart(backtest_dashboard, f'backtest_overview_{selected_run.name}')
        print(f"✓ Saved backtest_overview_{selected_run.name}")
except Exception as e:
    print(f"✗ Could not save backtest_dashboard: {e}")

print(f"\nCharts saved to: {viz.charts_dir}")